# Gov Collect Fabric

Tenant-wide read of the Fabric / Power BI control plane.

**Tier.** Runs at **T1** when the identity holds Fabric Administrator; falls
back to the workspaces the identity can see and records `tier="T0"` on the run
row so the app can say the inventory is partial. It never pretends.

**Not a security boundary.** Microsoft is explicit that tenant settings
"aren't a security measure". These rows are guardrail *evidence*; every
🟢 tenant-setting control is paired with a detective policy rule.

In [ ]:
dry_run = True
lakehouse_name = "governance_lh"
# Cap on workspaces expanded into items. 0 = no cap.
max_workspaces = 0

In [ ]:
# --- inlined from collectors/shape_common.py (unit-tested offline) ---
from __future__ import annotations

import json
import uuid
from datetime import datetime, timezone
from typing import Any, Callable, Iterable, Sequence


def utcnow() -> datetime:
    return datetime.now(timezone.utc)


def new_run_id() -> str:
    return str(uuid.uuid4())


def as_str(value: Any) -> str | None:
    """Normalise an API scalar to a string, preserving a real absence as None.

    Collector tables are all-string on purpose: every plane has its own id
    format, and coercing them into typed columns is how a join silently starts
    returning nothing.
    """
    if value is None:
        return None
    if isinstance(value, bool):
        return "true" if value else "false"
    if isinstance(value, (int, float)):
        return str(value)
    if isinstance(value, str):
        return value
    return json.dumps(value, ensure_ascii=False, sort_keys=True)


def as_json(value: Any) -> str | None:
    """Stable JSON for a blob column. Sorted keys so diffs are meaningful."""
    if value is None:
        return None
    return json.dumps(value, ensure_ascii=False, sort_keys=True, default=str)


def stamp(rows: Iterable[dict], run_id: str, scanned_at: datetime | None = None) -> list[dict]:
    """Attach the run provenance every `gov_actual_*` row carries."""
    when = scanned_at or utcnow()
    out = []
    for row in rows:
        enriched = dict(row)
        enriched["run_id"] = run_id
        enriched["scanned_at"] = when
        out.append(enriched)
    return out


class RunLedger:
    """Accumulates what a collector did, for the `gov_runs` row and the app.

    Errors are first-class: a collector that quietly drops an unreadable object
    produces a governance report that is wrong in the most dangerous direction —
    it under-reports access.
    """

    def __init__(self, collector: str, module: str, tier: str) -> None:
        self.run_id = new_run_id()
        self.collector = collector
        self.module = module
        self.tier = tier
        self.started_at = utcnow()
        self.finished_at: datetime | None = None
        self.errors: list[dict[str, str]] = []
        self.counts: dict[str, int] = {}

    def count(self, table: str, n: int) -> None:
        self.counts[table] = self.counts.get(table, 0) + n

    def error(self, scope: str, exc: BaseException | str) -> None:
        self.errors.append(
            {
                "scope": scope,
                "type": type(exc).__name__ if isinstance(exc, BaseException) else "Error",
                "message": str(exc),
            }
        )

    def finish(self) -> dict:
        self.finished_at = utcnow()
        return {
            "run_id": self.run_id,
            "collector": self.collector,
            "module": self.module,
            "tier": self.tier,
            "started_at": self.started_at,
            "finished_at": self.finished_at,
            "n_objects": sum(self.counts.values()),
            "n_errors": len(self.errors),
            "error_json": as_json(self.errors) if self.errors else None,
            "duration_s": (self.finished_at - self.started_at).total_seconds(),
        }

    def exit_value(self, *, dry_run: bool) -> dict:
        """Actuator-contract-shaped result (PLAN.md §14) for the app to parse."""
        return {
            "ok": True,
            "dry_run": dry_run,
            "run_id": self.run_id,
            "collector": self.collector,
            "module": self.module,
            "tier": self.tier,
            "counts": dict(self.counts),
            "n_errors": len(self.errors),
            "errors": self.errors[:20],
            "finished_at": (self.finished_at or utcnow()).isoformat(),
        }


def safe_each(
    items: Sequence[Any],
    fn: Callable[[Any], list[dict]],
    ledger: RunLedger,
    scope_of: Callable[[Any], str],
) -> list[dict]:
    """Map `fn` over `items`, recording per-item failures instead of raising."""
    rows: list[dict] = []
    for item in items:
        try:
            rows.extend(fn(item))
        except Exception as exc:  # noqa: BLE001 — a collector must never hard-fail
            ledger.error(scope_of(item), exc)
    return rows

In [ ]:
# --- inlined from collectors/runtime.py (unit-tested offline) ---
from __future__ import annotations

import json
import time
from typing import Any, Callable


class RestError(RuntimeError):
    def __init__(self, status: int, url: str, body: str) -> None:
        super().__init__(f"{status} {url}: {body[:400]}")
        self.status = status
        self.url = url


def fabric_client():
    """A `sempy` REST client for Fabric / Power BI, under the running identity."""
    import sempy.fabric as fabric  # type: ignore

    return fabric.FabricRestClient()


def rest_get(client, path: str, *, retries: int = 4) -> dict[str, Any]:
    """GET with backoff on 429/5xx.

    Admin APIs are rate-limited (25 req/min on some tenant-setting endpoints), and
    a nightly crawl that gives up on the first 429 silently under-reports — which
    is the worst possible failure mode for a governance inventory.
    """
    delay = 2.0
    last: Exception | None = None
    for _ in range(retries):
        response = client.get(path)
        if response.status_code == 200:
            return response.json() if response.text else {}
        if response.status_code in (429, 500, 502, 503, 504):
            retry_after = response.headers.get("Retry-After")
            time.sleep(float(retry_after) if retry_after else delay)
            delay = min(delay * 2, 60)
            last = RestError(response.status_code, path, response.text)
            continue
        raise RestError(response.status_code, path, response.text)
    raise last or RestError(0, path, "exhausted retries")


def graph_token(scope: str = "https://graph.microsoft.com/.default") -> str:
    """Delegated Graph token for the identity the notebook runs as."""
    import notebookutils  # type: ignore

    return notebookutils.credentials.getToken(scope)


def graph_get(token: str, url: str, *, retries: int = 4) -> dict[str, Any]:
    import urllib.error
    import urllib.request

    if not url.startswith("http"):
        url = f"https://graph.microsoft.com{url}"

    delay = 2.0
    for _ in range(retries):
        request = urllib.request.Request(url, headers={"Authorization": f"Bearer {token}"})
        try:
            with urllib.request.urlopen(request) as response:  # noqa: S310 - fixed host
                return json.loads(response.read().decode("utf-8"))
        except urllib.error.HTTPError as exc:
            if exc.code in (429, 500, 502, 503, 504):
                time.sleep(delay)
                delay = min(delay * 2, 60)
                continue
            raise RestError(exc.code, url, exc.read().decode("utf-8", "replace")) from exc
    raise RestError(0, url, "exhausted retries")


def graph_call(token: str, method: str, url: str, body: dict | None = None) -> dict[str, Any]:
    """Graph request with a method — the write-capable sibling of `graph_get`.

    Deliberately **not** retried on 5xx: a POST that may have partially applied
    must not be replayed blindly. The actuator's read-before-write makes a
    retry safe only after re-reading, and that is the caller's decision.
    """
    import urllib.error
    import urllib.request

    if not url.startswith("http"):
        url = f"https://graph.microsoft.com{url}"

    data = json.dumps(body).encode("utf-8") if body is not None else None
    request = urllib.request.Request(url, data=data, method=method.upper())
    request.add_header("Authorization", f"Bearer {token}")
    if data is not None:
        request.add_header("Content-Type", "application/json")

    try:
        with urllib.request.urlopen(request) as response:  # noqa: S310 - fixed host
            payload = response.read().decode("utf-8")
            return json.loads(payload) if payload else {}
    except urllib.error.HTTPError as exc:
        raise RestError(exc.code, url, exc.read().decode("utf-8", "replace")) from exc


def fabric_call(client, method: str, path: str, body: dict | None = None) -> dict[str, Any]:
    """Fabric REST with a method, through the `sempy` client.

    Same no-retry stance as `graph_call`, for the same reason.
    """
    verb = method.upper()
    if verb == "GET":
        response = client.get(path)
    elif verb == "POST":
        response = client.post(path, json=body or {})
    elif verb == "PATCH":
        response = client.patch(path, json=body or {})
    elif verb == "DELETE":
        response = client.delete(path)
    else:
        raise ValueError(f"unsupported method {method}")

    if response.status_code not in (200, 201, 202, 204):
        raise RestError(response.status_code, path, response.text)
    return response.json() if response.text else {}


def write_table(
    spark,
    lakehouse: str,
    table: str,
    rows: list[dict],
    *,
    dry_run: bool,
    log: Callable[[str, str, str], None],
) -> int:
    """Overwrite one `gov_actual_*` table with this run's rows.

    Overwrite, not append: these tables are a *snapshot of current reality*, and
    the run ledger plus `gov_audit` carry the history. An append-only actual-state
    table is how a drift engine starts comparing against last month.
    """
    if dry_run:
        log(table, "Planned", f"{len(rows)} rows")
        return len(rows)
    if not rows:
        log(table, "Skipped (no permission)", "no rows collected")
        return 0
    try:
        df = spark.createDataFrame(rows)
        df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
            f"{lakehouse}.{table}"
        )
        log(table, "Created", f"{len(rows)} rows")
        return len(rows)
    except Exception as exc:  # noqa: BLE001
        log(table, "Failed", f"{type(exc).__name__}: {exc}")
        return 0


def write_run_row(spark, lakehouse: str, summary: dict, *, dry_run: bool) -> None:
    if dry_run:
        return
    try:
        spark.createDataFrame([summary]).write.mode("append").option(
            "mergeSchema", "true"
        ).saveAsTable(f"{lakehouse}.gov_runs")
    except Exception as exc:  # noqa: BLE001
        print(f"gov_runs append failed: {exc}")


def finish(ledger, spark, lakehouse: str, *, dry_run: bool) -> str:
    summary = ledger.finish()
    write_run_row(spark, lakehouse, summary, dry_run=dry_run)
    result = ledger.exit_value(dry_run=dry_run)
    try:
        import notebookutils  # type: ignore

        notebookutils.notebook.exit(json.dumps(result))
    except ImportError:
        print(json.dumps(result, indent=2))
    return json.dumps(result)

In [ ]:
# --- inlined from collectors/shape_fabric.py (unit-tested offline) ---
from __future__ import annotations

from typing import Any



def shape_tenant_settings(payload: dict[str, Any]) -> list[dict]:
    """`GET /v1/admin/tenantsettings` → one row per setting.

    `scope` is the field a policy rule actually keys on:
      * `Everyone`        — enabled with no security-group restriction
      * `SecurityGroups`  — enabled for named groups only
      * `Excluded`        — enabled for everyone *except* named groups
      * `Disabled`        — off
    Microsoft is explicit that tenant settings are *not* a security boundary, so
    these rows are guardrail evidence, always paired with a detective rule.
    """
    rows: list[dict] = []
    for setting in payload.get("tenantSettings", []) or []:
        enabled = bool(setting.get("enabled"))
        included = setting.get("enabledSecurityGroups") or []
        excluded = setting.get("excludedSecurityGroups") or []

        if not enabled:
            scope = "Disabled"
        elif included:
            scope = "SecurityGroups"
        elif excluded:
            scope = "Excluded"
        else:
            scope = "Everyone"

        rows.append(
            {
                "setting_name": as_str(setting.get("settingName")),
                "title": as_str(setting.get("title")),
                "setting_group": as_str(setting.get("tenantSettingGroup")),
                "enabled": as_str(enabled),
                "scope": scope,
                "can_specify_security_groups": as_str(
                    setting.get("canSpecifySecurityGroups")
                ),
                "delegate_to_capacity": as_str(setting.get("delegateToCapacity")),
                "delegate_to_domain": as_str(setting.get("delegateToDomain")),
                "delegate_to_workspace": as_str(setting.get("delegateToWorkspace")),
                "enabled_groups_json": as_json(included),
                "excluded_groups_json": as_json(excluded),
                "properties_json": as_json(setting.get("properties")),
            }
        )
    return rows


def shape_capacity_overrides(payload: dict[str, Any]) -> list[dict]:
    """`GET /v1/admin/capacities/delegatedTenantSettingOverrides` → per-capacity rows."""
    rows: list[dict] = []
    for override in payload.get("value", []) or []:
        capacity_id = as_str(override.get("id"))
        for setting in override.get("tenantSettings", []) or []:
            rows.append(
                {
                    "capacity_id": capacity_id,
                    "setting_name": as_str(setting.get("settingName")),
                    "enabled": as_str(setting.get("enabled")),
                    "enabled_groups_json": as_json(
                        setting.get("enabledSecurityGroups") or []
                    ),
                }
            )
    return rows


def shape_workspaces(payload: dict[str, Any]) -> list[dict]:
    """Admin (`workspaces`) or user-scoped (`value`) workspace list."""
    source = payload.get("workspaces")
    if source is None:
        source = payload.get("value") or []
    rows: list[dict] = []
    for ws in source:
        rows.append(
            {
                "workspace_id": as_str(ws.get("id")),
                "workspace_name": as_str(ws.get("displayName") or ws.get("name")),
                "workspace_type": as_str(ws.get("type")),
                "capacity_id": as_str(ws.get("capacityId")),
                "state": as_str(ws.get("state")),
                "description": as_str(ws.get("description")),
            }
        )
    return rows


def shape_workspace_roles(workspace_id: str, payload: dict[str, Any]) -> list[dict]:
    """`GET /v1/workspaces/{id}/roleAssignments`.

    `principal_type` matters as much as the role: a *group* holding Contributor
    is an entitlement that can be compiled onto; a *user* holding it directly is
    almost always drift.
    """
    rows: list[dict] = []
    for assignment in payload.get("value", []) or []:
        principal = assignment.get("principal") or {}
        rows.append(
            {
                "workspace_id": as_str(workspace_id),
                "principal_id": as_str(principal.get("id")),
                "principal_type": as_str(principal.get("type")),
                "principal_name": as_str(principal.get("displayName")),
                "role": as_str(assignment.get("role")),
            }
        )
    return rows


#: Item types that carry their own tenant-level creation switch. Everything else
#: is governed only by the workspace role, which is exactly the "no per-item-type
#: role" gap the product exists to make visible.
TENANT_GATED_ITEM_TYPES = {
    "OrgApp",
    "FabricApp",
    "Ontology",
    "DigitalTwinBuilder",
    "Plan",
    "DeploymentPlan",
}


def shape_items(workspace: dict[str, Any], payload: dict[str, Any]) -> list[dict]:
    rows: list[dict] = []
    workspace_id = as_str(workspace.get("id"))
    workspace_name = as_str(workspace.get("displayName") or workspace.get("name"))
    for item in payload.get("value", []) or []:
        item_type = as_str(item.get("type"))
        rows.append(
            {
                "item_id": as_str(item.get("id")),
                "item_type": item_type,
                "item_name": as_str(item.get("displayName")),
                "workspace_id": workspace_id,
                "workspace_name": workspace_name,
                "description": as_str(item.get("description")),
                "is_tenant_gated": as_str(item_type in TENANT_GATED_ITEM_TYPES),
            }
        )
    return rows


def shape_org_apps(items_rows: list[dict]) -> list[dict]:
    """Org apps, projected out of the item inventory."""
    return [
        {
            "app_id": row["item_id"],
            "app_name": row["item_name"],
            "kind": "Fabric",
            "workspace_id": row["workspace_id"],
            "workspace_name": row["workspace_name"],
        }
        for row in items_rows
        if row.get("item_type") == "OrgApp"
    ]


def shape_orgapp_audiences(app: dict[str, Any], definition: dict[str, Any]) -> list[dict]:
    """`OrgAppAudience` children of an org app.

    The audience *objects* are API-manageable; **who is in an audience is not**
    — there is no documented public API for audience membership. So every row is
    stamped `membership_source='Portal-manual'` and `membership_known='false'`,
    and the app must never imply it knows who can see an org app.
    """
    rows: list[dict] = []
    for child in definition.get("parts", []) or []:
        path = str(child.get("path", ""))
        if not path.endswith(".OrgAppAudience"):
            continue
        rows.append(
            {
                "audience_id": as_str(child.get("id") or path),
                "audience_name": as_str(path.rsplit("/", 1)[-1].replace(".OrgAppAudience", "")),
                "app_id": as_str(app.get("app_id")),
                "workspace_id": as_str(app.get("workspace_id")),
                "membership_source": "Portal-manual",
                "membership_known": "false",
            }
        )
    return rows

In [ ]:
steps = []


def log(step, status, detail=""):
    steps.append({"step": step, "status": status, "detail": detail})
    print(f"[{status:>22}] {step}{(' — ' + detail) if detail else ''}")


ledger = RunLedger("Gov Collect Fabric", "fabric", "T1")
client = fabric_client()
print(f"run_id={ledger.run_id} dry_run={dry_run}")

## Tenant settings and capacity overrides

The first thing an admin wants to see, and the cheapest to get.

In [ ]:
tenant_setting_rows = []
try:
    payload = rest_get(client, "/v1/admin/tenantsettings")
    tenant_setting_rows = shape_tenant_settings(payload)
    ledger.count("gov_actual_tenant_settings", len(tenant_setting_rows))
except Exception as exc:  # noqa: BLE001
    # Losing admin read is not fatal — it downgrades the whole run to T0.
    ledger.tier = "T0"
    ledger.error("tenantsettings", exc)

capacity_override_rows = []
try:
    payload = rest_get(
        client, "/v1/admin/capacities/delegatedTenantSettingOverrides"
    )
    capacity_override_rows = shape_capacity_overrides(payload)
    ledger.count("gov_actual_capacity_overrides", len(capacity_override_rows))
except Exception as exc:  # noqa: BLE001
    ledger.error("capacityOverrides", exc)

## Workspaces

Admin list when we can, own list when we cannot. The `tier` on the run row is
what tells the app which of the two it is looking at.

In [ ]:
raw_workspaces = []
try:
    payload = rest_get(client, "/v1/admin/workspaces")
    raw_workspaces = payload.get("workspaces", []) or []
except Exception as exc:  # noqa: BLE001
    ledger.tier = "T0"
    ledger.error("adminWorkspaces", exc)
    try:
        payload = rest_get(client, "/v1/workspaces")
        raw_workspaces = payload.get("value", []) or []
    except Exception as inner:  # noqa: BLE001
        ledger.error("workspaces", inner)

workspace_rows = shape_workspaces({"workspaces": raw_workspaces})
ledger.count("gov_actual_workspaces", len(workspace_rows))
log("workspaces", "Created", f"{len(workspace_rows)} found, tier={ledger.tier}")

if max_workspaces and len(raw_workspaces) > max_workspaces:
    raw_workspaces = raw_workspaces[:max_workspaces]
    ledger.error("workspaces", f"capped at {max_workspaces}")

## Role assignments and items

One request per workspace, sequential. Parallel bursts against the admin APIs
are the fastest route to a 429, and a throttled crawl that gives up
under-reports access — the most dangerous direction to be wrong in.

In [ ]:
role_rows = []
item_rows = []

for ws in raw_workspaces:
    ws_id = ws.get("id")
    if not ws_id:
        continue
    try:
        payload = rest_get(client, f"/v1/workspaces/{ws_id}/roleAssignments")
        role_rows.extend(shape_workspace_roles(ws_id, payload))
    except Exception as exc:  # noqa: BLE001
        ledger.error(f"roles:{ws.get('displayName') or ws_id}", exc)
    try:
        payload = rest_get(client, f"/v1/workspaces/{ws_id}/items")
        item_rows.extend(shape_items(ws, payload))
    except Exception as exc:  # noqa: BLE001
        ledger.error(f"items:{ws.get('displayName') or ws_id}", exc)

ledger.count("gov_actual_workspace_roles", len(role_rows))
ledger.count("gov_actual_items", len(item_rows))
log("roles + items", "Created", f"{len(role_rows)} roles, {len(item_rows)} items")

## Org apps and audiences

The audience *objects* are API-manageable. **Who is in an audience is not** —
there is no documented public API for audience membership, so every audience
row is stamped `membership_known='false'` and the app must never imply it
knows who can see an org app.

In [ ]:
org_app_rows = shape_org_apps(item_rows)
audience_rows = []

for app in org_app_rows:
    try:
        payload = rest_get(
            client,
            f"/v1/workspaces/{app['workspace_id']}/items/{app['app_id']}/getDefinition",
        )
        definition = payload.get("definition", payload)
        audience_rows.extend(shape_orgapp_audiences(app, definition))
    except Exception as exc:  # noqa: BLE001
        ledger.error(f"orgapp:{app.get('app_name')}", exc)

ledger.count("gov_actual_orgapps", len(org_app_rows))
ledger.count("gov_actual_orgapp_audiences", len(audience_rows))
log("org apps", "Created", f"{len(org_app_rows)} apps, {len(audience_rows)} audiences")

## Write

In [ ]:
TABLES = [
    ("gov_actual_tenant_settings", tenant_setting_rows),
    ("gov_actual_capacity_overrides", capacity_override_rows),
    ("gov_actual_workspaces", workspace_rows),
    ("gov_actual_workspace_roles", role_rows),
    ("gov_actual_items", item_rows),
    ("gov_actual_orgapps", org_app_rows),
    ("gov_actual_orgapp_audiences", audience_rows),
]

for table, rows in TABLES:
    write_table(
        spark,  # noqa: F821 — provided by the Fabric runtime
        lakehouse_name,
        table,
        stamp(rows, ledger.run_id),
        dry_run=dry_run,
        log=log,
    )

finish(ledger, spark, lakehouse_name, dry_run=dry_run)  # noqa: F821